# 10 — Avaliação final (`evaluate`) em GPU (Google Colab / Kaggle)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/luanfreitas5/sentimento-ptbr-llm/blob/main/notebooks/10_cloud_gpu_evaluate.ipynb)

Executa a etapa **`evaluate`** (`pipelines.evaluate.run_evaluate_stage`): aplica no conjunto de teste todos os
modelos persistidos em `models/checkpoints/` (clássicos, deep learning, Transformers e LLMs), calcula métricas
com **IC por bootstrap**, compara os modelos com **McNemar + Holm**, avalia contra os *gold sets*, calibra
probabilidades, faz a análise por fatias/erros e (opcional) a ablação. O código é o do repositório; os
hiperparâmetros vêm de `configs/evaluation.yaml`. A GPU acelera a inferência dos Transformers e dos LLMs (Ollama).

## Antes de começar

| | Colab | Kaggle |
|---|---|---|
| GPU | Runtime → **T4 GPU** | Settings → **GPU T4 x2** ou **P100** |
| Dados | `treino/validacao/teste.parquet` (e, opcionalmente, `gold_*.parquet`) em `MyDrive/sentimento-ptbr-llm/data/gold/` | Dataset anexado com esses arquivos |
| Checkpoints | `*.joblib`, `*.pt` e `*.llm.json` em `MyDrive/sentimento-ptbr-llm/models/checkpoints/` | Dataset anexado com os checkpoints |
| Internet | já habilitada | Settings → **Internet On** (clona o repositório, instala o Ollama e baixa os modelos) |
| Resultados | Drive (`MyDrive/sentimento-ptbr-llm/outputs/`) | Aba **Output** (`/kaggle/working/outputs/`) |

**Primeiro rode com `SMOKE_TEST = True`** (poucos textos por LLM, poucas reamostragens, sem ablação) para validar
o pipeline. Os checkpoints devem ter sido gerados pelos notebooks 08/09 (ou localmente); copie para a pasta de
checkpoints acima os arquivos `checkpoints/*` das saídas desses notebooks.

> **Aviso:** checkpoints `.joblib` (scikit-learn/XGBoost) são pickles e dependem da versão da biblioteca. Se um
> arquivo não carregar (`input stream corrupted`, `ModelPersistenceError`), retreine-o no mesmo ambiente em que
> será avaliado, ou deixe-o fora de `MODEL_NAMES`.

## 1. Plataforma e GPU

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
PLATFORM = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"Plataforma detectada: {PLATFORM}")

import torch

if torch.cuda.is_available():
    gpu_properties = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu_properties.name} | VRAM: {gpu_properties.total_memory / 1e9:.1f} GB")
else:
    print(
        "ATENÇÃO: nenhuma GPU detectada. Colab: Runtime > Change runtime type > GPU. "
        "Kaggle: Settings > Accelerator > GPU T4 x2 / P100."
    )

## 2. Parâmetros da execução

In [ ]:
# --- Repositório -----------------------------------------------------------
REPO_URL = "https://github.com/luanfreitas5/sentimento-ptbr-llm.git"
REPO_BRANCH = "main"

# --- Onde estão os dados e os checkpoints -----------------------------------
# Colab : pastas do Google Drive (relativas a /content/drive/MyDrive).
# Kaggle: os arquivos devem estar em Datasets anexados (Add Input); o notebook procura
#         em /kaggle/input/* automaticamente.
DRIVE_DATA_DIR = "sentimento-ptbr-llm/data/gold"  # treino/validacao/teste.parquet e gold_*.parquet
DRIVE_CHECKPOINTS_DIR = "sentimento-ptbr-llm/models/checkpoints"  # *.joblib, *.pt, *.llm.json
DRIVE_OUTPUT_DIR = "sentimento-ptbr-llm/outputs"  # Colab: onde salvar os resultados

# --- Execução ---------------------------------------------------------------
NOTEBOOK_TAG = "evaluate"

MODEL_NAMES = None  # None = todos os checkpoints. Ex.: ["svm", "llama3_2", "bertimbau"]

SMOKE_TEST = True  # True: poucos textos por LLM, bootstrap curto, sem ablação. False: YAML.
SMOKE_SAMPLES = 30  # textos de teste enviados a cada LLM no smoke test
SMOKE_BOOTSTRAP = 100  # reamostragens do bootstrap no smoke test
SKIP_ABLATION = True  # a ablação re-treina modelos clássicos: use False só no run completo
LLM_MAX_TEST_SAMPLES = None  # None = configs/evaluation.yaml -> evaluate.llm_max_test_samples
RANDOM_SEED = 42
AUTO_DISCONNECT = False  # Colab: encerra o runtime ao terminar

## 3. Repositório, dados e checkpoints

In [ ]:
# --- Pastas de trabalho ----------------------------------------------------
WORK_DIR = Path("/kaggle/working") if IN_KAGGLE else Path("/content") if IN_COLAB else Path.cwd()
REPO_DIR = WORK_DIR / "sentimento-ptbr-llm"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DATA_SEARCH_DIRS = [Path("/content/drive/MyDrive") / DRIVE_DATA_DIR, Path("/content/data")]
    CHECKPOINT_SEARCH_DIRS = [Path("/content/drive/MyDrive") / DRIVE_CHECKPOINTS_DIR]
    OUTPUT_ROOT = Path("/content/drive/MyDrive") / DRIVE_OUTPUT_DIR
elif IN_KAGGLE:
    DATA_SEARCH_DIRS = [path for path in Path("/kaggle/input").glob("*") if path.is_dir()]
    CHECKPOINT_SEARCH_DIRS = DATA_SEARCH_DIRS
    OUTPUT_ROOT = Path("/kaggle/working/outputs")  # tudo em /kaggle/working vira output do notebook
else:
    DATA_SEARCH_DIRS = [REPO_DIR / "data" / "gold"]
    CHECKPOINT_SEARCH_DIRS = [REPO_DIR / "models" / "checkpoints"]
    OUTPUT_ROOT = WORK_DIR / "outputs"

# Separa execução de teste rápido (smoke) da execução completa: evita reaproveitar
# por engano um checkpoint treinado com poucos dados.
RUN_TAG = "smoke" if SMOKE_TEST else "full"
PERSIST_DIR = OUTPUT_ROOT / NOTEBOOK_TAG / RUN_TAG
PERSIST_DIR.mkdir(parents=True, exist_ok=True)
print(f"Resultados serão salvos em: {PERSIST_DIR}")


def get_secret(name: str) -> str | None:
    """Lê um segredo (Colab: Secrets; Kaggle: Add-ons > Secrets; senão, variável de ambiente)."""
    try:
        if IN_COLAB:
            from google.colab import userdata

            return userdata.get(name)
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient

            return UserSecretsClient().get_secret(name)
    except Exception:  # segredo inexistente ou sem acesso concedido ao notebook
        return None
    return os.environ.get(name)


# --- Clona (ou atualiza) o repositório --------------------------------------
if not (REPO_DIR / ".git").exists():
    token = get_secret("GITHUB_TOKEN")  # só necessário se o repositório for privado
    clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, clone_url, str(REPO_DIR)],
        check=True,
        capture_output=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], check=True
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
print(
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"],
        check=False,
        capture_output=True,
        text=True,
    ).stdout
)

# --- Copia os parquets de dados para data/gold do repositório ---------------
GOLD_DIR = REPO_DIR / "data" / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)
for file_name in ("treino.parquet", "validacao.parquet", "teste.parquet"):
    found = next(
        (match for base in DATA_SEARCH_DIRS if base.exists() for match in base.rglob(file_name)),
        None,
    )
    if found is None:
        raise FileNotFoundError(
            f"'{file_name}' não encontrado em {[str(p) for p in DATA_SEARCH_DIRS]}. "
            "Envie os parquets de data/gold/ para o Drive (Colab) ou crie um Dataset (Kaggle)."
        )
    if found.resolve() != (GOLD_DIR / file_name).resolve():
        shutil.copy2(found, GOLD_DIR / file_name)
    print(f"OK  {file_name}  <-  {found}")

sys.path.insert(0, str(REPO_DIR / "src"))  # o projeto importa módulos a partir de src/
os.chdir(REPO_DIR)

# --- Gold sets (opcionais): habilitam a avaliação contra as referências -------
gold_candidates = {
    match.name: match
    for base in DATA_SEARCH_DIRS
    if base.exists()
    for match in base.rglob("gold_*.parquet")
}
for gold_name, gold_file in sorted(gold_candidates.items()):
    if gold_file.resolve() != (GOLD_DIR / gold_name).resolve():
        shutil.copy2(gold_file, GOLD_DIR / gold_name)
    print(f"OK  {gold_name}  <-  {gold_file}")
if not gold_candidates:
    print("Nenhum gold_*.parquet encontrado: a avaliação contra os gold sets será pulada.")

# --- Checkpoints dos modelos (+ vetorizador TF-IDF dos modelos clássicos) ------
CHECKPOINTS_DIR = REPO_DIR / "models" / "checkpoints"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_files = {
    match.name: match
    for base in CHECKPOINT_SEARCH_DIRS
    if base.exists()
    for pattern in ("*.joblib", "*.pt", "*.llm.json")
    for match in base.rglob(pattern)
}
if not checkpoint_files:
    raise FileNotFoundError(
        "Nenhum checkpoint (*.joblib, *.pt, *.llm.json) em "
        f"{[str(p) for p in CHECKPOINT_SEARCH_DIRS]}. "
        "Copie os arquivos de models/checkpoints/ (ou os de checkpoints/ dos notebooks 08/09)."
    )
for file_name, found in sorted(checkpoint_files.items()):
    if found.resolve() != (CHECKPOINTS_DIR / file_name).resolve():
        shutil.copy2(found, CHECKPOINTS_DIR / file_name)
    print(f"OK  {file_name}  <-  {found}")

# Nomes de modelo Ollama exigidos pelos LLMs (lidos dos *.llm.json selecionados)
import json

OLLAMA_MODELS = sorted(
    {
        json.loads(spec.read_text(encoding="utf-8"))["overrides"]["model_name"]
        for spec in CHECKPOINTS_DIR.glob("*.llm.json")
        if MODEL_NAMES is None or spec.name.removesuffix(".llm.json") in MODEL_NAMES
    }
)
print(f"Modelos Ollama necessários: {OLLAMA_MODELS or 'nenhum'}")

## 4. Dependências Python

In [ ]:
import subprocess
import sys

PACKAGES = [
    "pydantic",
    "pydantic-settings",
    "python-dotenv",
    "pyyaml",
    "joblib",
    "pandas",
    "pandera[polars]",
    "polars",
    "pyarrow",
    "scikit-learn",
    "scipy",
    "statsmodels",
    "xgboost",
    "mlflow",
    "rich",
    "matplotlib",
    "seaborn",
    "transformers>=4.44",
    "accelerate>=1.0",
    "ollama",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)

## 5. Servidor Ollama (só se houver LLMs)
Instala o Ollama, inicia o servidor em segundo plano e baixa os modelos dos checkpoints `*.llm.json`.
Sem LLMs entre os checkpoints, esta seção não faz nada.

In [ ]:
import time
import urllib.request

OLLAMA_URL = "http://localhost:11434"


def is_ollama_up() -> bool:
    """Verifica se o servidor Ollama responde."""
    try:
        urllib.request.urlopen(OLLAMA_URL, timeout=2)
    except OSError:
        return False
    return True


if OLLAMA_MODELS and shutil.which("ollama") is None:
    subprocess.run(
        "apt-get -qq update && apt-get -qq install -y zstd pciutils", shell=True, check=True
    )
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

if OLLAMA_MODELS and not is_ollama_up():
    with (WORK_DIR / "ollama_server.log").open("w") as server_log:
        subprocess.Popen(["ollama", "serve"], stdout=server_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if is_ollama_up():
            break
        time.sleep(1)
    else:
        raise RuntimeError(f"Servidor Ollama não respondeu; veja {WORK_DIR / 'ollama_server.log'}")
print("Servidor Ollama ativo." if OLLAMA_MODELS else "Sem LLMs: Ollama não é necessário.")

for ollama_name in OLLAMA_MODELS:
    print(f"Baixando {ollama_name} ...")
    subprocess.run(["ollama", "pull", ollama_name], check=True)
if OLLAMA_MODELS:
    print(subprocess.run(["ollama", "list"], check=False, capture_output=True, text=True).stdout)

## 6. Configuração da avaliação
Lê `configs/evaluation.yaml` (como `src/main.py --stage evaluate`). No smoke test reduz o bootstrap, o número
de textos enviados a cada LLM e dispensa a ablação.

In [ ]:
import logging

from rich.logging import RichHandler

from config.constants import CONFIG_FILE_NAMES
from config.paths import CONFIGS_DIR, load_project_paths
from io_utils.yaml import read_yaml

logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    handlers=[RichHandler(rich_tracebacks=True)],
    force=True,
)

paths = load_project_paths()
evaluation_config = read_yaml(CONFIGS_DIR / CONFIG_FILE_NAMES["evaluation"])
classical_params = read_yaml(CONFIGS_DIR / CONFIG_FILE_NAMES["model_params"])["classical"]

ablation_config = dict(evaluation_config["ablation"])
ablation_config["model_params"] = {
    **classical_params.get(ablation_config["model"], {}),
    **(ablation_config.get("model_params") or {}),
}

n_bootstrap = SMOKE_BOOTSTRAP if SMOKE_TEST else evaluation_config["uncertainty"]["n_bootstrap"]
llm_max_test_samples = (
    SMOKE_SAMPLES
    if SMOKE_TEST
    else (LLM_MAX_TEST_SAMPLES or evaluation_config["evaluate"]["llm_max_test_samples"])
)
skip_ablation = SMOKE_TEST or SKIP_ABLATION

evaluate_kwargs = {
    "model_names": MODEL_NAMES,
    "n_bootstrap": n_bootstrap,
    "confidence_level": evaluation_config["uncertainty"]["confidence_level"],
    "alpha": evaluation_config["significance_tests"]["alpha"],
    "random_seed": RANDOM_SEED,
    "llm_max_test_samples": llm_max_test_samples,
    "ablation_config": ablation_config,
    "skip_ablation": skip_ablation,
    "calibration_config": (
        evaluation_config["calibration"]
        if evaluation_config["calibration"].get("enabled")
        else None
    ),
    "slice_config": (
        evaluation_config["slice_evaluation"]
        if evaluation_config["slice_evaluation"].get("enabled")
        else None
    ),
}
print({key: value for key, value in evaluate_kwargs.items() if key != "ablation_config"})

## 7. Etapa `evaluate`
Aplica cada checkpoint ao teste e grava tabelas, estatísticas, figuras e o resumo em `reports/`. A inferência dos
LLMs é a parte lenta (uma chamada por texto); a GPU do Ollama é confirmada em `ollama ps` (coluna PROCESSOR).

In [ ]:
from pipelines.evaluate import run_evaluate_stage

started_at = time.perf_counter()
evaluate_result = run_evaluate_stage(paths, **evaluate_kwargs)
print(f"Avaliação concluída em {(time.perf_counter() - started_at) / 60:.1f} min")
if OLLAMA_MODELS:
    print(subprocess.run(["ollama", "ps"], check=False, capture_output=True, text=True).stdout)

## 8. Resultados

In [ ]:
import polars as pl

with pl.Config(tbl_rows=50, tbl_cols=20, fmt_str_lengths=60):
    print("=== Métricas por modelo (com IC) ===")
    print(evaluate_result.metrics_table)
    if not evaluate_result.mcnemar_table.is_empty():
        print("=== McNemar (p ajustado por Holm) ===")
        print(evaluate_result.mcnemar_table)
    if evaluate_result.ablation_table is not None:
        print("=== Ablação ===")
        print(evaluate_result.ablation_table)
print(json.dumps(evaluate_result.summary.get("best_model", {}), indent=2, ensure_ascii=False))
print("Arquivos gerados:")
for name, file_path in evaluate_result.files.items():
    print(f"  {name:<12} {file_path}")

## 9. Salvamento
Copia `reports/` (tabelas, estatísticas, métricas, figuras, ablação) para a pasta persistente.

In [ ]:
persist_reports = PERSIST_DIR / "reports"
if persist_reports.exists():
    shutil.rmtree(persist_reports)
shutil.copytree(REPO_DIR / "reports", persist_reports)

archive_path = shutil.make_archive(
    str(WORK_DIR / f"resultados_{NOTEBOOK_TAG}_{RUN_TAG}"), "zip", PERSIST_DIR
)
print(f"Pacote com tabelas, estatísticas, figuras e métricas: {archive_path}")
print("Colab : os arquivos já estão no Drive. Kaggle: use Save Version e baixe em Output.")

if AUTO_DISCONNECT and IN_COLAB:
    from google.colab import runtime

    runtime.unassign()

## Próximos passos (no seu computador)

1. Copie o conteúdo de `reports/` baixado para o `reports/` do repositório (ou apenas compare com a execução local).
2. Com `SMOKE_TEST = True` as métricas **não** devem ser reportadas: rode de novo com `SMOKE_TEST = False`
   (e `SKIP_ABLATION = False` para incluir a ablação).
3. Gere figuras, Model Cards e o Datasheet com `make report`.